In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# MODEL 1

In [2]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [4]:
df.shape,df.isnull().sum()

((2000, 8),
 id        0
 prompt    0
 A         0
 B         0
 C         0
 D         0
 E         0
 answer    0
 dtype: int64)

In [5]:
df["answer"].value_counts()
#the classes are balanced.

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [6]:
train_df, val_df = train_test_split(df,test_size=0.2,random_state=42)
len(train_df),len(val_df)

(1600, 400)

In [7]:
train_df.to_csv("train_split.csv", index=False)
val_df.to_csv("val_split.csv", index=False)

In [8]:
def clean_text(text):
    """
    Cleans a text string.

    Steps:
    1. Convert to lowercase
    2. Remove extra spaces
    """

    text = str(text).lower()

    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    return text

In [9]:
def combine_text(row):

    text = f"""
Question: {row['prompt']}

A: {row['A']}

B: {row['B']}

C: {row['C']}

D: {row['D']}

E: {row['E']}
"""

    return clean_text(text)

In [10]:
train_df["text"] = train_df.apply(combine_text, axis=1)
val_df["text"] = val_df.apply(combine_text, axis=1)

In [11]:
train_df.apply(combine_text, axis=1)

968     question: pick the best possible answer: what ...
240     question: which of the following is correct? w...
819     question: choose the correct answer: what is t...
692     question: how do the lunar laser ranging exper...
420     question: determine the correct option: which ...
                              ...                        
1130    question: pick the best possible answer: what ...
1294    question: pick the best possible answer: what ...
860     question: which of the following is correct? w...
1459    question: determine the correct option: what i...
1126    question: choose the correct answer: what did ...
Length: 1600, dtype: object

In [12]:
print(train_df["text"].iloc[0])

question: pick the best possible answer: what is the proposed name for the field that is responsible for cosmic inflation and the metric expansion of space? among the listed options. a: inflaton b: quanta c: scalar d: metric e: conformal cyclic cosmology


In [13]:
from collections import Counter
class Vocabulary:
    def __init__(self, min_freq=1):
        self.min_freq = min_freq

        self.word2idx = {
            "<PAD>": 0,
            "<UNK>": 1
        }
        self.idx2word = {
            0: "<PAD>",
            1: "<UNK>"
        }
    def build_vocab(self, texts):
        counter = Counter()
        for text in texts:
            words = text.split()
            counter.update(words)
        idx = 2
        for word, freq in counter.items():
            if freq >= self.min_freq:
                self.word2idx[word] = idx
                self.idx2word[idx] = word
                idx += 1
    def numericalize(self, text):
    
        tokens = text.split()
    
        return [
            self.word2idx.get(word, self.word2idx["<UNK>"])
            for word in tokens
        ]
    def __len__(self):
        return len(self.word2idx)

In [14]:

vocab = Vocabulary()

vocab.build_vocab(train_df["text"])

print("Vocabulary Size:", len(vocab.word2idx))

Vocabulary Size: 3822


In [15]:

import torch
from torch.utils.data import Dataset

label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}


class MCQDataset(Dataset):

    def __init__(self, dataframe, vocab):

        self.df = dataframe.reset_index(drop=True)

        self.vocab = vocab

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        input_ids = self.vocab.numericalize(row["text"])

        label = label_map[row["answer"]]

        return input_ids, label

In [16]:

train_dataset = MCQDataset(train_df, vocab)
val_dataset = MCQDataset(val_df, vocab)
print(len(train_dataset))

sample = train_dataset[0]

print(sample)

1600
([2, 3, 4, 5, 6, 7, 8, 9, 4, 10, 11, 12, 4, 13, 14, 9, 15, 12, 16, 17, 18, 4, 19, 20, 21, 22, 23, 4, 24, 25, 26, 27, 28, 29, 30, 31, 32, 19, 33, 34, 35, 36], 0)


In [17]:
import torch
from torch.nn.utils.rnn import pad_sequence

In [18]:
def collate_fn(batch):

    inputs = []
    labels = []

    for input_ids, label in batch:
        inputs.append(torch.tensor(input_ids))
        labels.append(label)

    inputs = pad_sequence(
        inputs,
        batch_first=True,
        padding_value=0
    )

    labels = torch.tensor(labels)

    return inputs, labels

In [19]:
from torch.utils.data import DataLoader

In [20]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

In [21]:
inputs, labels = next(iter(train_loader))

print(inputs.shape)
print(labels.shape)

torch.Size([32, 418])
torch.Size([32])


In [22]:
import torch
import torch.nn as nn

class BiLSTMClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        output_dim,
        num_layers,
        dropout
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_dim * 2,
            output_dim
        )
    def forward(self, input_ids):

        embedded = self.embedding(input_ids)
    
        output, (hidden, cell) = self.lstm(embedded)
    
        hidden_forward = hidden[-2]
    
        hidden_backward = hidden[-1]
    
        hidden = torch.cat(
            (hidden_forward, hidden_backward),
            dim=1
        )
    
        hidden = self.dropout(hidden)
    
        logits = self.fc(hidden)
    
        return logits
        

In [23]:
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import accuracy_score, f1_score

In [24]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [25]:
model = BiLSTMClassifier(
    vocab_size= len(vocab),
    embedding_dim=128,
    hidden_dim=256,
    output_dim=5,
    num_layers=2,
    dropout=0.5
)

model = model.to(device)

In [26]:
criterion = nn.CrossEntropyLoss()


In [27]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [28]:
def map_at_3(y_true, y_pred_top3):
    """
    Computes Mean Average Precision at 3 (MAP@3)

    Parameters
    ----------
    y_true : list
        True labels
    y_pred_top3 : list
        Top-3 predicted class indices for each sample

    Returns
    -------
    float
    """

    score = 0.0

    for true_label, preds in zip(y_true, y_pred_top3):

        if true_label in preds:

            rank = preds.tolist().index(true_label) + 1

            score += 1 / rank

    return score / len(y_true)

In [29]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    total_loss = 0

    all_preds = []      # For Accuracy/F1
    all_top3 = []       # For MAP@3
    all_labels = []

    for inputs, labels in loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        top3 = torch.topk(outputs, k=3, dim=1).indices
        
        all_preds.extend(preds.cpu().numpy())
        all_top3.extend(top3.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    accuracy = accuracy_score(all_labels, all_preds)

    f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    map3 = map_at_3(all_labels, all_top3)
    
    avg_loss = total_loss / len(loader)
    
    return avg_loss, accuracy, f1, map3

    

In [30]:
def validate_one_epoch(model, loader, criterion, device):

    model.eval()

    total_loss = 0

    all_preds = []      # For Accuracy/F1
    all_top3 = []       # For MAP@3
    all_labels = []
    with torch.no_grad():

        for inputs, labels in loader:

            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)

            loss = criterion(outputs, labels)

            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            top3 = torch.topk(outputs, k=3, dim=1).indices
            
            all_preds.extend(preds.cpu().numpy())
            all_top3.extend(top3.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        accuracy = accuracy_score(all_labels, all_preds)
    
        f1 = f1_score(
            all_labels,
            all_preds,
            average="macro"
        )

        map3 = map_at_3(all_labels, all_top3)
        
        avg_loss = total_loss / len(loader)
        
        return avg_loss, accuracy, f1, map3
        
           

In [31]:
num_epochs = 10

best_f1 = 0

for epoch in range(num_epochs):
    train_loss, train_acc, train_f1, train_map3 = train_one_epoch(model,
        train_loader,
        criterion,
        optimizer,
        device)
   
    val_loss, val_acc, val_f1, val_map3 = validate_one_epoch(model,
        val_loader,
        criterion,
        device)
    
    print(f"Epoch {epoch+1}/{num_epochs}")

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Acc: {train_acc:.4f} | "
        f"F1: {train_f1:.4f} | "
        f"MAP@3: {train_map3:.4f}"
    )
    
    print(
        f"Val Loss: {val_loss:.4f} | "
        f"Acc: {val_acc:.4f} | "
        f"F1: {val_f1:.4f} | "
        f"MAP@3: {val_map3:.4f}"
    )

    if val_f1 > best_f1:

        best_f1 = val_f1

        torch.save(
            
            model.state_dict(),
            "best_model.pth"
        )

        print("Best model saved!")

Epoch 1/10
Train Loss: 1.6004 | Acc: 0.2406 | F1: 0.1571 | MAP@3: 0.4142
Val Loss: 1.5573 | Acc: 0.2825 | F1: 0.1619 | MAP@3: 0.4683
Best model saved!
Epoch 2/10
Train Loss: 1.3925 | Acc: 0.3600 | F1: 0.3216 | MAP@3: 0.5384
Val Loss: 1.3339 | Acc: 0.3500 | F1: 0.3186 | MAP@3: 0.5329
Best model saved!
Epoch 3/10
Train Loss: 1.4753 | Acc: 0.3538 | F1: 0.3548 | MAP@3: 0.5221
Val Loss: 1.2561 | Acc: 0.4300 | F1: 0.4169 | MAP@3: 0.6054
Best model saved!
Epoch 4/10
Train Loss: 0.9626 | Acc: 0.5713 | F1: 0.5661 | MAP@3: 0.7309
Val Loss: 0.8754 | Acc: 0.6550 | F1: 0.6338 | MAP@3: 0.8008
Best model saved!
Epoch 5/10
Train Loss: 0.4988 | Acc: 0.8113 | F1: 0.8019 | MAP@3: 0.8961
Val Loss: 0.4225 | Acc: 0.8725 | F1: 0.8701 | MAP@3: 0.9296
Best model saved!
Epoch 6/10
Train Loss: 0.2355 | Acc: 0.9356 | F1: 0.9357 | MAP@3: 0.9647
Val Loss: 0.2865 | Acc: 0.9475 | F1: 0.9545 | MAP@3: 0.9687
Best model saved!
Epoch 7/10
Train Loss: 0.0928 | Acc: 0.9812 | F1: 0.9823 | MAP@3: 0.9897
Val Loss: 0.0529 | Ac

In [32]:
model.to(device)
model.eval()

BiLSTMClassifier(
  (embedding): Embedding(3822, 128, padding_idx=0)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.5, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=512, out_features=5, bias=True)
)

In [33]:
import pickle

with open("vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

In [34]:
class TestDataset(Dataset):

    def __init__(self, df, vocab, max_len=128):
        self.df = df
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        text = self.df.iloc[idx]["prompt"]   # or "text" depending on your dataset

        tokens = self.vocab.numericalize(text)

        tokens = tokens[:self.max_len]

        if len(tokens) < self.max_len:
            tokens += [0] * (self.max_len - len(tokens))

        return torch.tensor(tokens)

In [35]:
test_dataset = TestDataset(test_df, vocab)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [36]:
id2label = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

In [37]:
model.eval()
all_top3 = []
with torch.no_grad():

    for inputs in test_loader:

        inputs = inputs.to(device)

        outputs = model(inputs)

        top3 = torch.topk(outputs, k=3, dim=1).indices

        all_top3.extend(top3.cpu().numpy())
all_top3[:5]

[array([0, 3, 2]),
 array([2, 0, 1]),
 array([1, 2, 4]),
 array([4, 0, 2]),
 array([4, 2, 0])]

In [38]:
predictions = []

for pred in all_top3:
    labels = [id2label[i] for i in pred]
    predictions.append(" ".join(labels))

In [39]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "prediction": predictions
})

In [40]:
submission.to_csv("submission.csv", index=False)

In [41]:
submission.head()

,id,prediction
0,1,A D C
1,2,C A B
2,3,B C E
3,4,E A C
4,5,E C A


# BASELINE MODEL

In [42]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [43]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [44]:
train.describe()

,id
count,2000.000000
mean,1000.500000
std,577.494589
min,1.000000
25%,500.750000
50%,1000.500000
75%,1500.250000
max,2000.000000


In [45]:
train["answer"].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [46]:
train.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


Dataset Observations

- Number of samples in training set: 2000
- Number of columns: 8 
- Missing values: null
- Question types: science related
- Answer distribution: balanced

In [47]:
train.shape

(2000, 8)

In [48]:
train.head(10)

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A
5,6,Identify the correct statement: What is the ef...,"An electric field, precisely aligned with the ...","A magnetic field, randomly aligned with the sp...","A magnetic field, precisely aligned with the s...","A gravitational field, randomly aligned with t...","A gravitational field, precisely aligned with ...",C
6,7,"Which of the following is correct? What is a ""...",A type of coffee that is made by boiling coffe...,A pattern left by a particle-laden liquid afte...,A type of coffee that is made by mixing instan...,A type of coffee that is made by pouring hot w...,A pattern left by a particle-laden liquid afte...,E
7,8,Select the most accurate option: What is the L...,The Liouville density is a probability distrib...,The Liouville density is a quasiprobability di...,The Liouville density is a bounded probability...,The Liouville density is a probability distrib...,The Liouville density is a probability distrib...,A
8,9,Select the most accurate option: What are the ...,"They are unknown, but possibilities include la...",They are known to be black holes and Preon stars.,They are only MACHOs.,They are clusters of brown dwarfs.,They are new particles such as RAMBOs.,A
9,10,Which of the following is correct? What is the...,"The Wigner function W(x, p) is the Wigner tran...","The Wigner function W(x, p) is a source functi...","The Wigner function W(x, p) is the derivative ...","The Wigner function W(x, p) represents the Ham...","The Wigner function W(x, p) is the time deriva...",A


In [49]:
question = train.loc[0, "prompt"]
question

"Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options."

In [50]:
def clean_text(text):
    text = str(text)          # Ensure it's a string
    text = text.lower()       # Lowercase
    text = text.strip()       # Remove extra spaces
    return text

In [51]:
train["prompt"] = train["prompt"].apply(clean_text)
test["prompt"] = test["prompt"].apply(clean_text)

In [52]:
option_columns = ["A", "B", "C", "D", "E"]
for col in option_columns:
    train[col] = train[col].apply(clean_text)
    test[col] = test[col].apply(clean_text)

In [53]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,pick the best possible answer: what is martin ...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...,B
1,2,what is accelerator-based light-ion fusion?,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,A
2,3,determine the correct option: what is the term...,blueshifting,redshifting,reddening,whitening,yellowing,C
3,4,select the most accurate option: what is marti...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...,B
4,5,identify the correct statement: what is the co...,"simultaneity is relative, meaning that two eve...","simultaneity is relative, meaning that two eve...","simultaneity is absolute, meaning that two eve...",simultaneity is a concept that applies only to...,simultaneity is a concept that applies only to...,A


In [54]:
train["question_length"] = train["prompt"].apply(len)
train["question_length"].describe()

count    2000.000000
mean      117.669500
std        44.408674
min        19.000000
25%        87.750000
50%       111.000000
75%       141.000000
max       337.000000
Name: question_length, dtype: float64

In [55]:
corpus = []

corpus.extend(train["prompt"].tolist())

for col in ["A","B","C","D","E"]:
    corpus.extend(train[col].tolist())

In [56]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=30000
)
vectorizer.fit(corpus)

TfidfVectorizer(max_features=30000, stop_words='english')

In [57]:
question_vectors = vectorizer.transform(train["prompt"])
question_vectors 

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 19542 stored elements and shape (2000, 2762)>

In [58]:
option_vectors = {}

for option in ["A","B","C","D","E"]:
    option_vectors[option] = vectorizer.transform(train[option])

In [59]:
predictions = []

for i in range(len(train)):

    question = question_vectors[i]

    scores = {}

    for option in ["A","B","C","D","E"]:

        scores[option] = cosine_similarity(
            question,
            option_vectors[option][i]
        )[0][0]

    sorted_scores = sorted(
        scores.items(),
        key=lambda x:x[1],
        reverse=True
    )

    top3 = " ".join(
        [x[0] for x in sorted_scores[:3]]
    )

    predictions.append(top3)

In [60]:
test_predictions = []

for i in range(len(test)):

    question = question_vectors[i]

    scores = {}

    for option in ["A","B","C","D","E"]:

        scores[option] = cosine_similarity(
            question,
            option_vectors[option][i]
        )[0][0]

    sorted_scores = sorted(
        scores.items(),
        key=lambda x:x[1],
        reverse=True
    )

    top3 = " ".join(
        [x[0] for x in sorted_scores[:3]]
    )

    test_predictions.append(top3)

In [61]:
#submission = pd.DataFrame()

#submission["id"] = test["id"]

#submission["Prediction"] = test_predictions

#submission.head()

# MILESTONE 1

In [62]:
import pandas as pd

sample = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
print(sample.head())
print(sample.columns)

   ID Prediction
0   1      A B C
1   2      A B C
2   3      A B C
3   4      A B C
4   5      A B C
Index(['ID', 'Prediction'], dtype='object')


In [63]:
import pandas as pd
import random

sample = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

choices = [
    'A B C',
    'A C B',
    'B A C',
    'B C A',
    'C A B',
    'C B A'
]

sample['Prediction'] = [random.choice(choices) for _ in range(len(sample))]
sample.to_csv('submission.csv', index=False)

In [64]:
import pandas as pd
import numpy as np
import string
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity


Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?  
*


In [65]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
freq = train["answer"].value_counts()

q1 = freq.max() + freq.min()
q1

814

After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv? 

In [66]:
translator = str.maketrans('', '', string.punctuation)
#The str.maketrans() method in Python creates a translation table (a dictionary-like mapping)
#that specifies how characters in a string should be replaced, mapped, or deleted. 
def clean_prompt(text):
    text = str(text).lower()
    text = text.translate(translator)
    return text

vocab = set()

for text in train["prompt"]:
    words = clean_prompt(text).split()
    vocab.update(words)

q2 = len(vocab)
q2

859

Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  

In [67]:
row1 = train.iloc[0]
words = clean_prompt(row1["prompt"]).split()
filtered = [w for w in words if w not in ENGLISH_STOP_WORDS]

q3 = len(filtered)
q3


13

Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?  

In [68]:
combined_docs = []

for _, row in train.iterrows():
    combined = " ".join([str(row["prompt"]),str(row["A"]),str(row["B"]),str(row["C"]),
        str(row["D"]),str(row["E"])])
    combined_docs.append(combined)
vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(combined_docs)

q4 = len(vectorizer.get_feature_names_out())
q4

2762

Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

In [69]:
prompt_vec = vectorizer.transform([row1["prompt"]])
a_vec = vectorizer.transform([row1["A"]])

sim = cosine_similarity(prompt_vec, a_vec)[0][0]

q5 = round(sim, 4)
q5


np.float64(0.272)

Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.   
*


In [70]:
correct = 0
for _, row in train.iterrows():

    p = vectorizer.transform([row["prompt"]])

    scores = {}

    for opt in ["A","B","C","D","E"]:
        o = vectorizer.transform([row[opt]])
        scores[opt] = cosine_similarity(p, o)[0][0]

    pred = max(scores, key=scores.get)

    if pred == row["answer"]:
        correct += 1

q6 = 100 * correct / len(train)
q6

13.55

 If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  
*


In [71]:
# Ground truth C
# Prediction C A B
q7 = 1.0
q7


1.0

If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?  

In [72]:
# Ground truth B
# Prediction D B E
q8 = 1/2
q8


0.5

The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [73]:
freq_order = freq.index.tolist()

top3 = freq_order[:3]

def apk(actual, pred):
    for i,p in enumerate(pred[:3]):
        if p == actual:
            return 1/(i+1)
    return 0

scores = [
    apk(ans, top3)
    for ans in train["answer"]
]

q9 = np.mean(scores)
q9

np.float64(0.42125)

The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  

In [74]:
scores = []

for _, row in train.iterrows():

    p = vectorizer.transform([row["prompt"]])

    sims = []

    for opt in ["A","B","C","D","E"]:
        o = vectorizer.transform([row[opt]])
        sims.append(
            (opt, cosine_similarity(p,o)[0][0])
        )

    sims.sort(key=lambda x:x[1], reverse=True)

    pred = [x[0] for x in sims[:3]]

    scores.append(
        apk(row["answer"], pred)
    )

q10 = np.mean(scores)
q10

np.float64(0.2961666666666667)

# Milestone 2

In [75]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

train = dataset["train"]

Generating train split: 0 examples [00:00, ? examples/s]

In [76]:
def combine(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

train = train.map(combine)

print(len(train[51]["combined_text"]))

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

614


In [77]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

30522


In [78]:
print(tokenizer.sep_token)
print(tokenizer.sep_token_id)

[SEP]
102


In [79]:
prompts = train["prompt"][:]

# or
# prompts = list(train["prompt"])

encodings = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encodings["input_ids"].shape)

torch.Size([2000, 128])


In [80]:
from transformers import AutoModel

model = AutoModel.from_pretrained("bert-base-uncased")

inputs = tokenizer(train[0]["prompt"], return_tensors="pt")

outputs = model(**inputs)

print(outputs.last_hidden_state.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])


In [81]:
cls = outputs.last_hidden_state[0,0]

print(round(cls[:5].sum().item(),4))

-1.2001


In [82]:
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

text = "Light-ion fusion is a technique."

inputs = tokenizer(text, return_tensors="pt")

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print(tokens)

outputs = model(**inputs)

attention = outputs.attentions[-1][0,0]

fusion_index = tokens.index("fusion")

print(round(attention[0,fusion_index].item(),4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
0.1025


In [83]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

prompt = train[0]["prompt"]
option = train[0]["B"]

embeddings = model.encode(
    [prompt, option],
    convert_to_tensor=True
)

score = util.cos_sim(
    embeddings[0],
    embeddings[1]
)

print(round(score.item(),4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.7658


In [84]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [85]:
labels = ["A","B","C","D","E"]
def map3_score(predictions, truths):
    total = 0

    for pred, truth in zip(predictions, truths):
        if truth in pred:
            rank = pred.index(truth) + 1
            total += 1/rank

    return total/len(truths)

In [86]:
tfidf_predictions = []

for row in train:

    texts = [
        row["prompt"],
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"]
    ]

    vectorizer = TfidfVectorizer()

    X = vectorizer.fit_transform(texts)

    sims = cosine_similarity(X[0], X[1:])[0]

    ranking = np.argsort(sims)[::-1]

    top3 = [labels[i] for i in ranking[:3]]

    tfidf_predictions.append(top3)

In [87]:
minilm_predictions = []

for row in train:

    texts = [
        row["prompt"],
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"]
    ]

    emb = model.encode(
        texts,
        convert_to_tensor=True
    )

    sims = util.cos_sim(
        emb[0],
        emb[1:]
    )[0].cpu().numpy()

    ranking = np.argsort(sims)[::-1]

    top3 = [labels[i] for i in ranking[:3]]

    minilm_predictions.append(top3)

In [88]:
answers = train["answer"]

tfidf_map3 = map3_score(
    tfidf_predictions,
    answers
)

minilm_map3 = map3_score(
    minilm_predictions,
    answers
)

print("TFIDF MAP@3:", round(tfidf_map3,4))
print("MiniLM MAP@3:", round(minilm_map3,4))

TFIDF MAP@3: 0.3269
MiniLM MAP@3: 0.4231


In [89]:
count = 0

for tfidf, mini, ans in zip(
    tfidf_predictions,
    minilm_predictions,
    answers
):

    if ans not in tfidf and ans in mini:
        count += 1

print("Count:", count)

Count: 564


In [90]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification"
)

result = classifier(
    train[1]["prompt"],
    candidate_labels=[
        train[1]["A"],
        train[1]["B"],
        train[1]["C"]
    ]
)

print(result)

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

In [91]:
result2 = classifier(
    train[1]["prompt"],
    candidate_labels=[
        train[1]["A"],
        train[1]["B"],
        train[1]["C"]
    ],
    multi_label=True
)

softmax_sum = sum(result["scores"])
sigmoid_sum = sum(result2["scores"])

print(abs(softmax_sum-sigmoid_sum))

0.999490372636501


In [92]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-small",
    tokenizer="google/flan-t5-small"
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

In [93]:
prompt = f"""Question: {train[0]['prompt']}. Is the correct answer A: {train[0]['A']} or B: {train[0]['B']}? Answer with just the letter A or B."""

result = generator(
    prompt,
    max_new_tokens=5
)

print(result)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B."}]


In [94]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [95]:
prompt = f"""Question: {train[0]['prompt']}. Is the correct answer A: {train[0]['A']} or B: {train[0]['B']}? Answer with just the letter A or B."""

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=5
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)

B


# MILESTONE 3

In [1]:
!pip install faiss-cpu #install FAISS

import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

print("Creating nowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 64.2 MB/s eta 0:00:00:00:0100:01
Creating nowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


In [4]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
result = zs(prompt_150, candidate_labels=labels_150)

correct_option = ans_150
score = result["scores"][result["labels"].index(correct_option)]

print(round(score, 3))

0.384


In [6]:
query_embedding = model.encode([prompt_150])

D, I = index.search(query_embedding, 10)

retrieved_indices = I[0]

true_doc = kb[150]

for rank, idx in enumerate(retrieved_indices, start=1):
    if kb[idx] == true_doc:
        print(rank)

10


In [7]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

docs_10 = [kb[i] for i in retrieved_indices]

pairs = [[prompt_150, doc] for doc in docs_10]

scores = cross_encoder.predict(pairs)

ranking = np.argsort(scores)[::-1]

true_doc = kb[150]

for rank, idx in enumerate(ranking, start=1):
    if docs_10[idx] == true_doc:
        print(rank)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

1


In [8]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

row = train.iloc[42]

prompt = str(row["prompt"])

query_embedding = model.encode([prompt])

_, I = index.search(query_embedding, 5)

docs = [kb[i] for i in I[0]]

context = " ".join(docs)

rag = f"Context: {context} Question: {prompt}"

tokens = tokenizer(rag, truncation=False)

print(len(tokens["input_ids"]))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

216


In [9]:
true_doc = kb[150]

rag = f"Context: {true_doc} Question: {prompt_150}"

result = zs(rag, candidate_labels=labels_150)

score = result["scores"][result["labels"].index(ans_150)]

print(round(score,3))

0.989


In [10]:
wrong_doc = kb[999]

rag = f"Context: {wrong_doc} Question: {prompt_150}"

result = zs(rag, candidate_labels=labels_150)

score = result["scores"][result["labels"].index(ans_150)]

print(round(score,3))

0.529


In [11]:
hits = 0

for i in range(100):

    row = train.iloc[i]

    prompt = str(row["prompt"])

    correct_doc = str(row[row["answer"]])

    query_embedding = model.encode([prompt])

    _, I = index.search(query_embedding, 5)

    docs = [kb[idx] for idx in I[0]]

    if any(correct_doc in doc for doc in docs):
        hits += 1

hit_rate = hits / 100 * 100

print(round(hit_rate,1))

73.0


In [12]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

scores_all = []

for i in range(20):

    row = train.iloc[i]

    prompt = str(row["prompt"])

    labels = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    correct_letter = row["answer"]

    embedding = model.encode([prompt])

    _, I = index.search(embedding, 5)

    docs = [kb[idx] for idx in I[0]]

    pairs = [[prompt, d] for d in docs]

    ce_scores = cross_encoder.predict(pairs)

    best_doc = docs[np.argmax(ce_scores)]

    rag = f"Context: {best_doc} Question: {prompt}"

    result = zs(rag, candidate_labels=labels)

    ranked = result["labels"]

    letter_map = {
        str(row["A"]): "A",
        str(row["B"]): "B",
        str(row["C"]): "C",
        str(row["D"]): "D",
        str(row["E"]): "E"
    }

    predicted_letters = [letter_map[x] for x in ranked[:3]]

    if correct_letter == predicted_letters[0]:
        ap = 1
    elif correct_letter == predicted_letters[1]:
        ap = 1/2
    elif correct_letter == predicted_letters[2]:
        ap = 1/3
    else:
        ap = 0

    scores_all.append(ap)

print(round(np.mean(scores_all),3))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


0.975
